# 00 · Setup

1. Run the **next cell** once — that creates the widgets at the top of the notebook.
2. Set **catalog** to an existing Unity Catalog catalog you can `USE`. Schema may stay `metlife_claims`.
3. Run All (or run the remaining cells).

- This notebook does **not** create a catalog.
- Run on **serverless**. Do not create a classic cluster.

Every other notebook starts with `%run ./00_setup`, so the same widgets apply there.


In [ ]:
dbutils.widgets.text("catalog", "")
dbutils.widgets.text("schema", "metlife_claims")
dbutils.widgets.dropdown("create_schema_if_missing", "true", ["true", "false"])
print("Widgets ready — set catalog at the top, then run the next cell.")


In [ ]:
import re

CATALOG = dbutils.widgets.get("catalog").strip().strip("`")
SCHEMA = dbutils.widgets.get("schema").strip().strip("`")
CREATE_SCHEMA = dbutils.widgets.get("create_schema_if_missing") == "true"

IDENT = re.compile(r"^[A-Za-z_][A-Za-z0-9_-]*$")
assert CATALOG, (
    "Set the catalog widget to an existing catalog you can USE. "
    "Leave the default blank on purpose — do not guess a catalog name."
)
assert SCHEMA, "Set the schema widget (default metlife_claims is fine)."
assert IDENT.match(CATALOG) and IDENT.match(SCHEMA), (
    "catalog and schema must be Unity Catalog identifiers "
    "(start with a letter or underscore; letters, digits, underscore only)."
)
print(f"Target: {CATALOG}.{SCHEMA}")


In [ ]:
# Schema + volume: best-effort. Catalog create is intentionally omitted.
def uc(name: str) -> str:
    return "`" + name.replace("`", "") + "`"

if CREATE_SCHEMA:
    try:
        spark.sql(f"CREATE SCHEMA IF NOT EXISTS {uc(CATALOG)}.{uc(SCHEMA)}")
        print(f"Schema ready: {CATALOG}.{SCHEMA}")
    except Exception as e:
        print(f"Could not CREATE SCHEMA (ok if it already exists and you lack CREATE): {e}")

try:
    spark.sql(f"CREATE VOLUME IF NOT EXISTS {uc(CATALOG)}.{uc(SCHEMA)}.claim_docs")
    print(f"Volume ready: /Volumes/{CATALOG}/{SCHEMA}/claim_docs")
except Exception as e:
    print(f"Could not CREATE VOLUME (ok if already provisioned): {e}")

spark.sql(f"USE CATALOG {uc(CATALOG)}")
spark.sql(f"USE SCHEMA {uc(SCHEMA)}")
print("Current catalog/schema:", spark.sql("SELECT current_catalog(), current_schema()").collect()[0])

VOLUME = f"/Volumes/{CATALOG}/{SCHEMA}/claim_docs"
PDF_DIR = f"{VOLUME}/pdfs"
print(f"Volume path: {VOLUME}")


If the `USE` cell fails, you do not have access to that catalog/schema. Stop and get the account team to grant `USE CATALOG` / `USE SCHEMA` / `CREATE TABLE` before continuing.
